In [ ]:
import xarray as xr
from scipy.ndimage import map_coordinates

from typing import Union, List, Tuple
import os
import sys
import argparse
from pathlib import Path
import numpy as np


In [ ]:
class InitialConditionDataLoader :
    """
    Class to load initial condition data from NetCDF files using xarray.
    """
    
    def __init__(self, 
                 data_path : Union[str, List[str]], 
                 target_idx : int,
                 climato_path : str, 
                 rand_ds_path : str) -> None :
        
        self.data_path = data_path
        self.target_idx = target_idx
        self.climato_path = climato_path
        self.rand_ds_path = rand_ds_path
        
        # ---
        self.dataset = self.load_dataset(data_path)
        self.uv_depth_t0, self.uv_depth_t1 = self.get_ocean_current(self.dataset, target_idx)
        
    def load_dataset(self, 
                     data_path : Union[str, List[str]]) -> xr.Dataset :
            # Load the dataset using xarray
            chunks={"time": 15,  "lat": 96, "lon": 96}
            
            if isinstance(self.data_path, list):
                ds = xr.open_mfdataset(data_path, chunks=chunks)
                ds = ds.isel(time = slice(self.target_idx - 1, self.target_idx + 1)) # [2, ch, lat, lon] dataset
            else:
                ds = xr.open_dataset(data_path, chunks=chunks)
                ds = ds.isel(time = slice(self.target_idx - 1, self.target_idx + 1))
            
            return ds

    def get_ocean_current(self, 
                          ds : xr.Dataset, 
                          target_idx : int) -> tuple[xr.DataArray, xr.DataArray] :
        """
        Extract ocean current data (u and v components) from the dataset.

        Args:
            ds (xr.Dataset): Input dataset containing ocean current data.
            target_idx (int): Index of the base time step 't', not t-1 neither t+1.
            
        Returns:
            None: Saving u and v components as dictionary numpy arrays.
        """

        # Make indexing for time steps t-1, t
        idx_t0 = target_idx - 1
        idx_t1 = target_idx
        
        # Extract max, min and mean values of u and v components regarding to glonet input format 
        uv_t0 = ds["data"].isel(time=idx_t0, ch=slice(3, 5))
        uv_t1 = ds["data"].isel(time=idx_t1, ch=slice(3, 5))
        
        # Surface level
        max_uv_t0 = uv_t0.max(dim=("lat","lon"), skipna=True)
        min_uv_t0 = uv_t0.min(dim=("lat","lon"), skipna=True)
        mean_uv_t0 = uv_t0.mean(dim=("lat","lon"), skipna=True)
        std_uv_t0 = uv_t0.std(dim=("lat","lon"), skipna=True)

        # Initialize dictionary to store u, v components at different depth levels
        uv_depth_t0 = {"d0": uv_t0, 
                    "max_d0": max_uv_t0, 
                    "min_d0": min_uv_t0, 
                    "mean_d0": mean_uv_t0, 
                    "std_d0": std_uv_t0}
        
        # Work on depth levels 1-20
        for depth in range(1, 11):  # T, s, u, v depth states for depth levels 1-10
            uv_depth_da = ds["data"].isel(time=idx_t0, ch=[25 + depth, 34 + depth])
            uv_depth_t0[f"d{depth}"] = uv_depth_da
            uv_depth_t0[f"max_d{depth}"] = uv_depth_da.max(dim=("lat","lon"), skipna=True)
            uv_depth_t0[f"min_d{depth}"] = uv_depth_da.min(dim=("lat","lon"), skipna=True)
            uv_depth_t0[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat","lon"), skipna=True)
            uv_depth_t0[f"std_d{depth}"] = uv_depth_da.std(dim=("lat","lon"), skipna=True)
        for depth in range(11, 21):  # T, s, u, v depth states for depth levels 11-20
            uv_depth_da = ds["data"].isel(time=idx_t0, ch=[55 + depth, 64 + depth])
            uv_depth_t0[f"d{depth}"] = uv_depth_da
            uv_depth_t0[f"max_d{depth}"] = uv_depth_da.max(dim=("lat","lon"), skipna=True)
            uv_depth_t0[f"min_d{depth}"] = uv_depth_da.min(dim=("lat","lon"), skipna=True)
            uv_depth_t0[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat","lon"), skipna=True)
            uv_depth_t0[f"std_d{depth}"] = uv_depth_da.std(dim=("lat","lon"), skipna=True)
            
        # Same process for time step t1
        # Surface level
        max_uv_t1 = uv_t1.max(dim=("lat","lon"), skipna=True)
        min_uv_t1 = uv_t1.min(dim=("lat","lon"), skipna=True)
        mean_uv_t1 = uv_t1.mean(dim=("lat","lon"), skipna=True)
        std_uv_t1 = uv_t1.std(dim=("lat","lon"), skipna=True)

        # Initialize dictionary to store u, v components at different depth levels
        uv_depth_t1 = {"d0": uv_t1, 
                    "max_d0": max_uv_t1, 
                    "min_d0": min_uv_t1, 
                    "mean_d0": mean_uv_t1, 
                    "std_d0": std_uv_t1}
        
        # Work on depth levels 1-20
        for depth in range(1, 11):  # T, s, u, v depth states for depth levels 1-10
            uv_depth_da = ds["data"].isel(time=idx_t1, ch=[25 + depth, 34 + depth])
            uv_depth_t1[f"d{depth}"] = uv_depth_da
            uv_depth_t1[f"max_d{depth}"] = uv_depth_da.max(dim=("lat","lon"), skipna=True)
            uv_depth_t1[f"min_d{depth}"] = uv_depth_da.min(dim=("lat","lon"), skipna=True)
            uv_depth_t1[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat","lon"), skipna=True)
            uv_depth_t1[f"std_d{depth}"] = uv_depth_da.std(dim=("lat","lon"), skipna=True)
        for depth in range(11, 21):  # T, s, u, v depth states for depth levels 11-20
            uv_depth_da = ds["data"].isel(time=idx_t1, ch=[55 + depth, 64 + depth])
            uv_depth_t1[f"d{depth}"] = uv_depth_da
            uv_depth_t1[f"max_d{depth}"] = uv_depth_da.max(dim=("lat","lon"), skipna=True)
            uv_depth_t1[f"min_d{depth}"] = uv_depth_da.min(dim=("lat","lon"), skipna=True)
            uv_depth_t1[f"mean_d{depth}"] = uv_depth_da.mean(dim=("lat","lon"), skipna=True)
            uv_depth_t1[f"std_d{depth}"] = uv_depth_da.std(dim=("lat","lon"), skipna=True)
        
        return uv_depth_t0, uv_depth_t1
    
    def get_ssh_anomaly(self) -> xr.DataArray :
        """
        Extract sea surface height anomaly data from the dataset.

        Args:
            None
            
        Returns:
            xr.DataArray: Sea surface height anomaly data.
        """
        
        # Load data
        climato = xr.open_dataset(self.climato_path)
        rand_ds = xr.open_dataset(self.rand_ds_path)
        
        anomaly_ssh = climato.isel(time=0, ch=0)['data'] - rand_ds.isel(time=0, ch=0)['data']
        
        return anomaly_ssh

In [ ]:
import numpy as np


In [ ]:
class StochasticNoiseGenerator :
    """
    Class to generate stochastic noise fields for perturbations.
    Amplitude is pre-defined in method noise.
    """
    
    def __init__(self,
                 ssh_anomaly : xr.DataArray,
                 is_pointwise : bool, 
                 seed : int = 42) -> None :
        
        self.is_pointwise = is_pointwise
        self.seed = seed
        self.anomaly_ssh = ssh_anomaly
        
        self.nx = self.anomaly_ssh.sizes['lon']
        self.ny = self.anomaly_ssh.sizes['lat']
        
        # ---
        self.ocean_regions = self.ocean_region_definiition()
        self.noise = self.create_noise(self.nx, 
                                       self.ny, 
                                       self.ocean_regions, 
                                       self.seed)
        

    def ocean_region_definiition(self) :
        """
        Define ocean regions with different correlation lengths.
        This is a placeholder function and should be customized based on actual ocean regions.
        
        Returns:
        --------
        ocean_regions : list of dict
            List of region specifications with correlation lengths
        """
        ocean_regions = [
            # Gulf Stream (North Atlantic: 30-45°N, 80-50°W)
            {'name': 'Gulf Stream',
            'type': 'box', 
            'correlation_length': 4.0,  # Large mesoscale eddies
            'coords': {'y_range': [382, 542], 
                        'x_range': [300, 620]},
            'blend': True, 
            'blend_width': 70},
            
            # Kuroshio (North Pacific: 30-45°N, 135-160°E)
            {'name': 'Kuroshio',
            'type': 'box', 
            'correlation_length': 4.0,  # Large western boundary current eddies
            'coords': {'y_range': [382, 542], 
                        'x_range': [1160, 1460]},
            'blend': True, 
            'blend_width': 70},
            
            # Agulhas Current (South Indian Ocean: 45-30°S, 15-40°E)
            {'name': 'Agulhas',
            'type': 'box', 
            'correlation_length': 4.0,  # Strong retroflection eddies
            'coords': {'y_range': [82, 242], 
                        'x_range': [680, 980]},
            'blend': True, 
            'blend_width': 70},
            
            # Antarctic Circumpolar Current (60-45°S, circumpolar)
            {'name': 'Antarctic ACC',
            'type': 'box', 
            'correlation_length': 10.0,  # Strong ACC dynamics
            'coords': {'y_range': [80, 162], 
                        'x_range': [0, self.nx]},  # Circumpolar band
            'blend': True, 
            'blend_width': 30},
        ]
        
        return ocean_regions
    
    @staticmethod
    def create_correlation_length_field(shape, regions, default):
        """
        Create a field specifying correlation length at each location.
        
        Parameters:
        -----------
        shape : tuple
            Shape of the output field (ny, nx)
        regions : list of dict
            List of region specifications with correlation lengths
        default : background correlation length
        
        Returns:
        --------
        L_field : ndarray
            2D field with spatially-varying correlation length
        """
        ny, nx = shape
        L_field = np.ones((ny, nx)) * default  # Default correlation length
        
        y, x = np.mgrid[0:ny, 0:nx]
        
        for region in regions:
            mask = np.zeros((ny, nx))
            reg_type = region['type']
            correlation_length = region.get('correlation_length', 20.0)
            blend = region.get('blend', True)
            blend_width = region.get('blend_width', 30)
            
            if reg_type == 'box':
                coords = region['coords']
                y_min, y_max = coords['y_range']
                x_min, x_max = coords['x_range']
                
                if blend:
                    from scipy.ndimage import distance_transform_edt
                    box_mask = ((y >= y_min) & (y < y_max) & 
                            (x >= x_min) & (x < x_max)).astype(float)
                    dist_inside = distance_transform_edt(box_mask)
                    mask = np.minimum(1.0, dist_inside / blend_width)
                else:
                    mask = ((y >= y_min) & (y < y_max) & 
                        (x >= x_min) & (x < x_max)).astype(float)
            
            elif reg_type == 'circle':
                coords = region['coords']
                cy, cx = coords['center']
                radius = coords['radius']
                dist = np.sqrt((y - cy)**2 + (x - cx)**2)
                
                if blend:
                    mask = np.clip(1.0 - (dist - radius) / blend_width, 0, 1)
                else:
                    mask = (dist <= radius).astype(float)
            
            elif reg_type == 'gradient':
                # Linear gradient from one edge to another
                coords = region['coords']
                direction = coords.get('direction', 'x')  # 'x' or 'y'
                L_start = coords.get('L_start', 10.0)
                L_end = coords.get('L_end', 100.0)
                
                if direction == 'x':
                    gradient = np.linspace(L_start, L_end, nx)
                    L_field = np.tile(gradient, (ny, 1))
                else:  # 'y'
                    gradient = np.linspace(L_start, L_end, ny)
                    L_field = np.tile(gradient[:, np.newaxis], (1, nx))
                continue  # Don't blend with mask
            
            # Blend this region's correlation length with existing field
            L_field = L_field * (1 - mask) + correlation_length * mask
        
        return L_field
    @staticmethod
    def spatially_varying_gaussian_filter(shape, correlation_length_field, amplitude=1.0, seed=None):
        """
        Generate correlated noise with spatially-varying correlation length.
        Uses a multi-scale approach: generates noise at different scales and blends them.
        
        Parameters:
        -----------
        shape : tuple
            Shape of the output noise field (ny, nx)
        correlation_length_field : ndarray
            2D field specifying correlation length at each location
        amplitude : float
            Standard deviation of the output noise field
        seed : int or None
            Random seed for reproducibility
        
        Returns:
        --------
        correlated_noise : ndarray
            Noise field with spatially-varying correlation structure
        """
        from scipy.ndimage import gaussian_filter
        
        ny, nx = shape
        
        # Get the range of correlation lengths (ignoring NaN)
        L_min = np.nanmin(correlation_length_field)
        L_max = np.nanmax(correlation_length_field)
        
        # Create a set of base scales (logarithmically spaced)
        n_scales = 5
        L_scales = np.logspace(np.log10(max(L_min, 1)), np.log10(L_max), n_scales)
        
        # Generate ONE white noise field and filter it at different scales
        # This ensures reproducibility with the seed parameter
        if seed is not None :
            rng = np.random.RandomState(seed)
            white_noise = rng.randn(ny, nx)
        else :
            white_noise = np.random.randn(ny, nx)
        
        noise_scales = []
        for L in L_scales:
            filtered = gaussian_filter(white_noise, sigma=L)
            noise_scales.append(filtered)
        
        # Blend the different scales based on local correlation length
        output = np.zeros((ny, nx))
        
        for i, L in enumerate(L_scales):
            # Weight function: higher weight where correlation_length_field matches this scale
            # Using Gaussian weighting in log-space
            log_L = np.log(L)
            log_L_field = np.log(np.maximum(correlation_length_field, 1e-6))
            
            # Weight peaks at scales matching the local correlation length
            weight = np.exp(-((log_L_field - log_L)**2) / (2 * 0.5**2))
            
            output += weight * noise_scales[i]
        
        # Normalize weights (divide by sum of weights at each location)
        total_weight = np.zeros((ny, nx))
        for i, L in enumerate(L_scales):
            log_L = np.log(L)
            log_L_field = np.log(np.maximum(correlation_length_field, 1e-6))
            weight = np.exp(-((log_L_field - log_L)**2) / (2 * 0.5**2))
            total_weight += weight
        
        output = output / (total_weight + 1e-10)
        
        # Restore NaN where correlation_length_field was NaN
        output = np.where(np.isnan(correlation_length_field), np.nan, output)
        
        # Normalize to desired amplitude (only using valid values)
        output = output / np.nanstd(output) * amplitude / 10
        
        return output
    @staticmethod
    
    def spectral_correlated_noise(shape, correlation_length, amplitude=1.0, spectral_slope=-3, seed=None):
        """
        Generate spatially correlated noise using spectral method (more physical for ocean/eddies).
        
        Parameters:
        -----------
        shape : tuple
            Shape of the output noise field (ny, nx)
        correlation_length : float
            Correlation length scale in grid points
        amplitude : float
            Standard deviation of the output noise field
        spectral_slope : float
            Spectral slope (default -3 for turbulent flow, -5/3 for inertial range)
        seed : int or None
            Random seed for reproducibility
        
        Returns:
        --------
        correlated_noise : ndarray
            Spatially correlated noise field with realistic spectral characteristics
        """
        ny, nx = shape[:2]
        
        # Create frequency grid
        kx = np.fft.fftfreq(nx, d=1.0)
        ky = np.fft.fftfreq(ny, d=1.0)
        KX, KY = np.meshgrid(kx, ky)
        K = np.sqrt(KX**2 + KY**2)
        K[0, 0] = 1.0  # Avoid division by zero
        
        # Generate white noise in spectral space
        if seed is not None:
            rng = np.random.RandomState(seed)
            white_spectrum = np.fft.fft2(rng.randn(ny, nx))
        else:
            white_spectrum = np.fft.fft2(np.random.randn(ny, nx))
        
        # Apply power law spectrum filter
        # E(k) ~ k^spectral_slope with exponential cutoff at correlation scale
        k0 = 1.0 / correlation_length
        spectral_filter = K**spectral_slope * np.exp(-K / k0)
        spectral_filter[0, 0] = 0  # Remove mean
        
        # Apply filter in spectral space
        filtered_spectrum = white_spectrum * spectral_filter
        
        # Transform back to physical space
        correlated_noise = np.real(np.fft.ifft2(filtered_spectrum))
        
        # Normalize to desired amplitude
        correlated_noise = correlated_noise / np.std(correlated_noise) * amplitude
        
        # Make noise positive and map to [0, 1]
        correlated_noise_positive = np.exp(correlated_noise)
        adjust_factor = (4+ spectral_slope) + 1e-6  # Prevent division by zero
        correlated_noise_positive_normalized = correlated_noise_positive / adjust_factor
        
        return correlated_noise_positive_normalized

    def create_noise(self, 
                     nx : int, 
                     ny : int, 
                     ocean_regions : dict,
                     seed : int) -> np.ndarray :
        """Create stochasticity of the perturbation
        
        Args:
            nx (int): Number of grid points in x direction (latitude)
            ny (int): Number of grid points in y direction (longitude) 
            ocean_regions (dict): Ocean region definitions with correlation lengths
            seed (int): Random seed for reproducibility
        """
        
        if not self.is_pointwise :
            # For one vector perturbation. region-varying stochasticity 
            L_field_regions = self.create_correlation_length_field((self.ny, self.nx), ocean_regions, 50)

            # Create amplitude field based on SSH anomaly magnitude
            # Higher SSH variability = higher perturbation amplitude
            ssh_std = float(np.nanstd(self.anomaly_ssh.values))
            amp_field_regions = (np.abs(self.anomaly_ssh.values)) / ssh_std   # Scale to reasonable amplitude

            # Generate spatially-varying noise
            regional_noise = self.spatially_varying_gaussian_filter((self.ny, self.nx), 
                                                                    L_field_regions, 
                                                                    amplitude=0.1, 
                                                                    seed=seed)
            
            # Scale by amplitude field (based on SSH anomaly)
            regional_noise_scaled = regional_noise * amp_field_regions
            
            return regional_noise_scaled
        else : 
            # For pointwise displacement perturbation. Spectral cutoff spatially    correlated stochasticity
                # Use unique seed for each member
            seed = 4000 + seed
            rng = np.random.default_rng(seed)
            
            # Generate unique spectral slope k for this member
            k = rng.normal(loc=-1.0, scale=0.3)
            
            # Generate spectral correlated noise
            noise_spectral = self.spectral_correlated_noise((self.ny, self.nx), 
                                                            correlation_length=10, 
                                                            amplitude=0.3, 
                                                            spectral_slope=k,
                                                            seed=42)
            

            return noise_spectral

In [ ]:
class CurrentBasedPerturbationGenerator :
    """
    Class to make perturbation of initial condition based on ocean current data.
    """
    
    def __init__(self, 
                 data_path : Union[str, List[str]], 
                 target_idx : int,
                 climato_path : str,
                 rand_ds_path : str,
                 dt : float,
                 seed : int,

                 is_pointwise : bool, 
                 is_same_noise_in_time : bool,
                 is_same_noise_for_component : bool) -> None :

        self.dt = dt
        self.seed = seed
        self.is_pointwise = is_pointwise       
        self.is_same_noise_in_time = is_same_noise_in_time
        self.is_same_noise_for_component = is_same_noise_for_component

        # ---
        ic_dataloader = InitialConditionDataLoader(data_path, target_idx, climato_path, rand_ds_path)
        self.dataset = ic_dataloader.dataset
        self.uv_depth_t0 = ic_dataloader.uv_depth_t0
        self.uv_depth_t1 = ic_dataloader.uv_depth_t1
        self.ssh_anomaly = ic_dataloader.get_ssh_anomaly()


    
    @staticmethod        
    def create_land_mask_from_channel(data_array_2d : xr.DataArray, 
                                      threshold=1e-10) :
        """
        Create a land mask from a single channel data array
        """
        
        # data_array should be 2D 
        if len(data_array_2d.dims) == 2:  # (lat, lon)
            sample_data = data_array_2d
        else:  
            raise ValueError("Input data_array must be 2D (lat, lon).")
        
        land_mask = np.isnan(sample_data) | (np.abs(sample_data) < threshold)
        
        return land_mask
    @staticmethod
    def displace_array(data : np.ndarray, 
                       dx: Union[float, np.ndarray],
                       dy: Union[float, np.ndarray], 
                       land_mask: xr.DataArray) -> np.ndarray :
        """
        displacement function:
        - Replace each pixel with value from (x + dx, y + dy) with interpolation
        - If displaced coordinates indicate the land, retrieve original ocean value
        """
        
        ny, nx = data.shape
        
        # Create coordinate grids
        y_coords, x_coords = np.mgrid[0:ny, 0:nx]
        
        # Calculate new coordinates 
        # displacement could be float for onevector perturbation or array for pointwise perturbation
        new_y_coords = y_coords.astype(float) + dy
        new_x_coords = x_coords.astype(float) + dx
        
        # Apply displacement using linear interpolation
        displaced_data = map_coordinates(
            data,
            [new_y_coords, new_x_coords],
            order=1,           # Linear interpolation
            mode='reflect',  
            cval=np.nan,
            prefilter=False
        )
        
        # If displaced coordinates indicate the land (NaN), use original ocean value
        # Make a mask which contains invalid logic (displaced land pixel)
        #   and replace it with original data
        land_mask_array = land_mask.values
        invalid_mask = land_mask_array | np.isnan(displaced_data)
        displaced_data[invalid_mask] = data[invalid_mask]
        
        return displaced_data
    
    def displace_for_channels(self, 
                              dx_t0 : np.ndarray, 
                              dy_t0 : np.ndarray, 
                              dx_t1 : np.ndarray,
                              dy_t1 : np.ndarray) -> xr.DataArray :
        """Displace for every channel in the dataset.
        
        Return:
            xr.DataArray: Displaced dataarray with same format as input dataset.
        """
        
        # Initialize displaced dataarrays 
        displaced_datarray_t0 = []
        displaced_datarray_t1 = []
        
        # Displace for every ch
        for ch in range(self.dataset.isel(time=1).sizes['ch']) :
            
            # Create land mask
            dataarray_2d_t0 = self.dataset["data"].isel(time=0, ch=ch)
            dataarray_2d_t1 = self.dataset["data"].isel(time=1, ch=ch)
            self.land_mask = self.create_land_mask_from_channel(dataarray_2d_t0)
            
            # Displacing
            displaced_array_t0 = self.displace_array(dataarray_2d_t0.values, 
                                                    dx_t0, 
                                                    dy_t0, 
                                                    self.land_mask)
            displaced_array_t1 = self.displace_array(dataarray_2d_t1.values,
                                                    dx_t1, 
                                                    dy_t1, 
                                                    self.land_mask)

            displaced_datarray_t0.append(displaced_array_t0)
            displaced_datarray_t1.append(displaced_array_t1)
            
        # Make DataArray from displaced arrays
        displaced_da_t0 = xr.DataArray(data=np.stack(displaced_datarray_t0, axis=0), 
                                       dims=["ch", "lat", "lon"], 
                                       coords={"ch": self.dataset["data"].ch, 
                                               "lat": self.dataset["data"].lat, 
                                               "lon": self.dataset["data"].lon})
        displaced_da_t1 = xr.DataArray(data=np.stack(displaced_datarray_t1, axis=0), 
                                       dims=["ch", "lat", "lon"], 
                                       coords={"ch": self.dataset["data"].ch, 
                                               "lat": self.dataset["data"].lat, 
                                               "lon": self.dataset["data"].lon})
        
        # Combine t0 and t1 with time dimension to make input format
        discplace_input_sequence = xr.concat([displaced_da_t0, displaced_da_t1], dim="time")
        
        return discplace_input_sequence
      
    def generate_one_vector_perturbation(self, 
                                         seed : int,
                                         extraction_depth : int = 0) -> xr.DataArray  :
        """Generate perturbations from four different fixed vectors based on ocean current data. 

        Args:
            extraction_depth (int): Depth level with range(0, 21) to generate perturbation for. 
            0 for only surface , 20 for whole depth max and min value extraction.
        
        Retruns: None
            Save peturbated ocean satate in dict{np.array}
        """
    

        # Define four vectors in dictionary (min,max)
        uv_combinations_t0 = []
        uv_combinations_t1 = []

        for depth in range(extraction_depth + 1) : # range(21) : 
            # Process t0 (time = t-1)
            # Get min and max values for u and v at this depth level
            min_u = self.uv_depth_t0[f"min_d{depth}"].isel(ch=0).values  # u component
            max_u = self.uv_depth_t0[f"max_d{depth}"].isel(ch=0).values  # u component
            min_v = self.uv_depth_t0[f"min_d{depth}"].isel(ch=1).values  # v component
            max_v = self.uv_depth_t0[f"max_d{depth}"].isel(ch=1).values  # v component
            
            # Create 4 combinations for this depth level
            combinations_for_depth_t0 = [
                (min_u, min_v, depth, "min_u_min_v"),
                (max_u, min_v, depth, "max_u_min_v"),
                (min_u, max_v, depth, "min_u_max_v"),
                (max_u, max_v, depth, "max_u_max_v")
            ]

            # Process t1 (time = t)
            min_u_t1 = self.uv_depth_t1[f"min_d{depth}"].isel(ch=0).values  # u component
            max_u_t1 = self.uv_depth_t1[f"max_d{depth}"].isel(ch=0).values  # u component
            min_v_t1 = self.uv_depth_t1[f"min_d{depth}"].isel(ch=1).values  # v component
            max_v_t1 = self.uv_depth_t1[f"max_d{depth}"].isel(ch=1).values  # v component
            

            combinations_for_depth_t1 = [
                (min_u_t1, min_v_t1, depth, "min_u_min_v"),
                (max_u_t1, min_v_t1, depth, "max_u_min_v"),
                (min_u_t1, max_v_t1, depth, "min_u_max_v"),
                (max_u_t1, max_v_t1, depth, "max_u_max_v")
            ]

            uv_combinations_t0.extend(combinations_for_depth_t0)
            uv_combinations_t1.extend(combinations_for_depth_t1)
            
        # Gnerate noise field
        if self.is_same_noise_for_component :
            if self.is_same_noise_in_time :
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed).noise
                noise_v_t1 , noise_u_t0, noise_v_t0 = noise_u_t1.copy(), noise_u_t1.copy(), noise_u_t1.copy()
            else :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed).noise
                noise_v_t0 = noise_u_t0
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed + 1).noise
                noise_v_t1 = noise_u_t1
        else :
            if self.is_same_noise_in_time :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed).noise
                noise_v_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed).noise
                noise_u_t1 = noise_u_t0.copy()
                noise_v_t1 = noise_v_t0.copy()
            else :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed).noise
                noise_v_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed + 1).noise
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed + 2).noise
                noise_v_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=False, seed=seed + 3).noise
            
        # Iterate for each combination at t0
        for ((u_val_t0, v_val_t0, depth, combo_name), 
             (u_val_t1, v_val_t1, depth, combo_name)) in zip(uv_combinations_t0, 
                                                             uv_combinations_t1) :
            
            grid_shape = (self.uv_depth_t1[f"d{depth}"].sizes['lat'], self.uv_depth_t1[f"d{depth}"].sizes['lon'])
            u_2d_t0 = np.full(grid_shape, u_val_t0, dtype=np.float32)
            v_2d_t0 = np.full(grid_shape, v_val_t0, dtype=np.float32)
            u_2d_t1 = np.full(grid_shape, u_val_t1, dtype=np.float32)
            v_2d_t1 = np.full(grid_shape, v_val_t1, dtype=np.float32)
            
            dx_field_t0 = self.dt * (u_2d_t0 + noise_u_t0)
            dy_field_t0 = -1 * self.dt * (v_2d_t0 + noise_v_t0)
            dx_field_t1 = self.dt * (u_2d_t1 + noise_u_t1)
            dy_field_t1 = -1 * self.dt * (v_2d_t1 + noise_v_t1)
            
            one_vector_perturbated_state = self.displace_for_channels(dx_field_t0, 
                                                                      dy_field_t0, 
                                                                      dx_field_t1, 
                                                                      dy_field_t1)
            
        return one_vector_perturbated_state

    def generate_pointwise_1_perturbation(self, 
                                        seed : int) -> xr.DataArray :
        """Generate perturbations from pointwise displacement.
        Only considering surface dnymaics and applying to whole depth levels.
        """

        # Gnerate noise field
        if self.is_same_noise_for_component :
            if self.is_same_noise_in_time :
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t1 , noise_u_t0, noise_v_t0 = noise_u_t1.copy(), noise_u_t1.copy(), noise_u_t1.copy()
            else :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t0 = noise_u_t0
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 1).noise
                noise_v_t1 = noise_u_t1
        else :
            if self.is_same_noise_in_time :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 1).noise
                noise_u_t1 = noise_u_t0.copy()
                noise_v_t1 = noise_v_t0.copy()
            else :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 1).noise
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 2).noise
                noise_v_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 3).noise
        
        # Iterate for surface depth only
        for depth in range(1) :
            
            dx_field_t0 = self.dt * self.uv_depth_t0[f"d{depth}"].isel(ch=0).values * noise_u_t0
            dy_field_t0 = -1 * self.dt * self.uv_depth_t0[f"d{depth}"].isel(ch=1).values * noise_v_t0
            dx_field_t1 = self.dt * self.uv_depth_t1[f"d{depth}"].isel(ch=0).values * noise_u_t1
            dy_field_t1 = -1 * self.dt * self.uv_depth_t1[f"d{depth}"].isel(ch=1).values * noise_v_t1
        
            # Displacment coordinate mapping for all channels
            pointwise1_perturbated_state = self.displace_for_channels(dx_field_t0, 
                                                                      dy_field_t0, 
                                                                      dx_field_t1,
                                                                      dy_field_t1)
            
        return pointwise1_perturbated_state 

    def displace_channels_by_depth(self, 
                                   depth, 
                                   dx_t0, 
                                   dy_t0, 
                                   dx_t1, 
                                   dy_t1) -> Tuple[dict, dict] :
        """
        Apply displacement to all channels at a given depth level
        Returns a dictionary with displaced data for each channel
        """
        
        displaced_channels_t0 = {}
        displaced_channels_t1 = {}
        
        # Determine which channels belong to this depth level
        if depth == 0:
            # Surface: SSH (0), T (1), S (2), u (3), v (4)
            channels = [0, 1, 2, 3, 4]
            channel_names = ['ssh', 'thetao', 'so', 'uo', 'vo']
        elif 1 <= depth <= 10:
            # Depth 1-10: T, S, u, v
            base_t = 5 + depth - 1
            base_s = base_t + 10
            base_u = base_s + 10
            base_v = base_u + 10
            channels = [base_t, base_s, base_u, base_v]
            channel_names = ['thetao', 'so', 'uo', 'vo']
        else:  # 11 <= depth <= 20
            # Depth 11-20: T, S, u, v
            base_t = 35 + depth - 1
            base_s = base_t + 10
            base_u = base_s + 10
            base_v = base_u + 10
            channels = [base_t, base_s, base_u, base_v]
            channel_names = ['thetao', 'so', 'uo', 'vo']
        
        # Apply displacement to each channel
        for ch, ch_name in zip(channels, channel_names) :
            dataarray_2d_t0 = self.dataset["data"].isel(time=0, ch=ch).values
            dataarray_2d_t1 = self.dataset["data"].isel(time=1, ch=ch).values
    
            land_mask = self.create_land_mask_from_channel(dataarray_2d_t1)
              
            displaced_channels_t0[f'ch_{ch}'] = self.displace_array(dataarray_2d_t0, dx_t0, dy_t0, land_mask)
            displaced_channels_t1[f'ch_{ch}'] = self.displace_array(dataarray_2d_t1, dx_t1, dy_t1, land_mask)

        return displaced_channels_t0, displaced_channels_t1

    def generate_pointwise_2_perturbation(self, 
                                        seed : int) -> xr.DataArray :
        """Generate perturbations from pointwise displacement.
        Only considering every depth dynamics.
        """
        
        # Gnerate noise field
        if self.is_same_noise_for_component :
            if self.is_same_noise_in_time :
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t1 , noise_u_t0, noise_v_t0 = noise_u_t1.copy(), noise_u_t1.copy(), noise_u_t1.copy()
            else :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t0 = noise_u_t0
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 1).noise
                noise_v_t1 = noise_u_t1
        else :
            if self.is_same_noise_in_time :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 1).noise
                noise_u_t1 = noise_u_t0.copy()
                noise_v_t1 = noise_v_t0.copy()
            else :
                noise_u_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed).noise
                noise_v_t0 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 1).noise
                noise_u_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 2).noise
                noise_v_t1 = StochasticNoiseGenerator(self.ssh_anomaly, is_pointwise=True, seed=seed + 3).noise
                
        # Initialize displaced dataarrays
        displaced_arr_list_t0 = []
        displaced_arr_list_t1 = []
        
        # Iterate for every depth levels 
        for depth in range(21) :
            
            dx_field_t0 = self.dt * self.uv_depth_t0[f"d{depth}"].isel(ch=0).values * noise_u_t0
            dy_field_t0 = -1 * self.dt * self.uv_depth_t0[f"d{depth}"].isel(ch=1).values * noise_v_t0
            dx_field_t1 = self.dt * self.uv_depth_t1[f"d{depth}"].isel(ch=0).values * noise_u_t1
            dy_field_t1 = -1 * self.dt * self.uv_depth_t1[f"d{depth}"].isel(ch=1).values * noise_v_t1

            # Displacement depth by depth
            # In light of nature of dataset, 
            #   it should be sorted and converted in dataarray from outside of the displacing function.
            displace_by_ch_t0, displace_by_ch_t1 = self.displace_channels_by_depth(depth, 
                                                                                   dx_field_t0, 
                                                                                   dy_field_t0, 
                                                                                   dx_field_t1, 
                                                                                   dy_field_t1)
            
            displaced_arr_list_t0.append(displace_by_ch_t0)
            displaced_arr_list_t1.append(displace_by_ch_t1)
        # Combine all displaced channels from all depths
        # Each item in displaced_arr_list is a dict with keys like 'ch_0', 'ch_1', etc.
        all_displaced_t0 = {}
        all_displaced_t1 = {}
        for depth_dict in displaced_arr_list_t0:
            all_displaced_t0.update(depth_dict)
        for depth_dict in displaced_arr_list_t1:
            all_displaced_t1.update(depth_dict)
        
        sorted_channel = sorted(self.dataset["data"].ch.values.tolist())
        sorted_arr_t0 = np.stack([all_displaced_t0[f"ch_{ch}"] for ch in sorted_channel], axis=0)
        sorted_arr_t1 = np.stack([all_displaced_t1[f"ch_{ch}"] for ch in sorted_channel], axis=0)
        
        # Make DataArray from displaced arrays
        displaced_da_t0 = xr.DataArray(data=sorted_arr_t0, 
                                       dims=["ch", "lat", "lon"], 
                                       coords={"ch": self.dataset["data"].ch, 
                                               "lat": self.dataset["data"].lat, 
                                               "lon": self.dataset["data"].lon})
        displaced_da_t1 = xr.DataArray(data=sorted_arr_t1, 
                                       dims=["ch", "lat", "lon"], 
                                       coords={"ch": self.dataset["data"].ch, 
                                               "lat": self.dataset["data"].lat, 
                                               "lon": self.dataset["data"].lon})

        # Combine t0 and t1 with time dimension to make input format        
        pointwise2_perturbated_state = xr.concat([displaced_da_t0, displaced_da_t1], dim="time")            

        return  pointwise2_perturbated_state

# Generate ensemble

## Config ##


In [ ]:
# Configuration parameters
DATA_PATH = "/Odyssey/public/glonet/glorys12_1993-01-01_to_1993-06-30_init_states/combined_input.nc"
TARGET_IDX = 1
CLIMATO_PATH = "/Odyssey/public/glonet/1993-06-01_climatology/mean1.nc"
RAND_DS_PATH = "/Odyssey/public/glonet/2022-06-01_init_states/input1.nc"
OUTPUT_DIR = "./ensemble_output"
DT = 1.0
BASE_SEED = 42

# Create output directories
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

onevector_dir = output_dir / "one_vector"
pointwise1_dir = output_dir / "pointwise_1"
pointwise2_dir = output_dir / "pointwise_2"

for dir_path in [onevector_dir, pointwise1_dir, pointwise2_dir]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Data path: {DATA_PATH}")
print(f"  Target index: {TARGET_IDX}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Base seed: {BASE_SEED}")
print(f"\nOutput directories created:")
print(f"  - {onevector_dir}")
print(f"  - {pointwise1_dir}")
print(f"  - {pointwise2_dir}")

## Method 1: One Vector Perturbation (40 members)

Generates 40 ensemble members using:
- 4 (u,v) combinations from min/max values × 10 different seeds
- Regional Gaussian noise with SSH anomaly scaling

In [ ]:
print("="*80)
print("METHOD 1: ONE VECTOR PERTURBATION - 40 MEMBERS")
print("="*80)

member_count = 0

for combo_idx in range(4):  # 4 combinations (min/max u,v)
    print(f"\nProcessing combination {combo_idx + 1}/4...")
    
    for seed_offset in range(10):  # 10 different seeds per combination
        seed = BASE_SEED + combo_idx * 10 + seed_offset
        member_idx = member_count
        
        print(f"  Member {member_idx:03d}: seed={seed}", end="", flush=True)
        
        try:
            # Initialize generator
            generator = CurrentBasedPerturbationGenerator(
                data_path=DATA_PATH,
                target_idx=TARGET_IDX,
                climato_path=CLIMATO_PATH,
                rand_ds_path=RAND_DS_PATH,
                dt=DT,
                seed=seed,
                is_pointwise=False,
                is_same_noise_in_time=True,
                is_same_noise_for_component=True
            )
            
            # Generate perturbation
            perturbed_state = generator.generate_one_vector_perturbation(
                seed=seed,
                extraction_depth=0
            )
            
            # Save to file
            output_path = onevector_dir / f"member_{member_idx:03d}_initial_condition.nc"
            output_path.parent.mkdir(parents=True, exist_ok=True)
            
            ds_out = xr.Dataset({"data": perturbed_state})
            ds_out.to_netcdf(output_path)
            
            print(" ✓ Saved")
            member_count += 1
            
        except Exception as e:
            print(f" ✗ Error: {e}")
            continue

print(f"\n✓ Completed: {member_count} one_vector members generated")

## Method 2: Pointwise 1 Perturbation (80 members)

Generates 80 ensemble members using:
- Surface ocean currents with spectral correlated noise
- Surface displacement applied to all depth levels

In [ ]:
print("="*80)
print("METHOD 2: POINTWISE 1 PERTURBATION - 80 MEMBERS")
print("="*80)

member_count = 0

for member_idx in range(80):
    seed = BASE_SEED + 1000 + member_idx  # Offset to avoid seed collision
    
    print(f"Member {member_idx:03d}: seed={seed}", end="", flush=True)
    
    try:
        # Initialize generator
        generator = CurrentBasedPerturbationGenerator(
            data_path=DATA_PATH,
            target_idx=TARGET_IDX,
            climato_path=CLIMATO_PATH,
            rand_ds_path=RAND_DS_PATH,
            dt=DT,
            seed=seed,
            is_pointwise=True,
            is_same_noise_in_time=False,
            is_same_noise_for_component=False
        )
        
        # Generate perturbation
        perturbed_state = generator.generate_pointwise_1_perturbation(seed=seed)
        
        # Save to file
        output_path = pointwise1_dir / f"member_{member_idx:03d}_initial_condition.nc"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        ds_out = xr.Dataset({"data": perturbed_state})
        ds_out.to_netcdf(output_path)
        
        print(" ✓ Saved")
        member_count += 1
        
    except Exception as e:
        print(f" ✗ Error: {e}")
        continue
    
    # Progress indicator
    if (member_idx + 1) % 20 == 0:
        print(f"  [{member_count}/80 completed]")

print(f"\n✓ Completed: {member_count} pointwise_1 members generated")

## Method 3: Pointwise 2 Perturbation (80 members)

Generates 80 ensemble members using:
- Depth-specific ocean currents (21 depth levels)
- Spectral correlated noise applied at each depth independently

In [ ]:
print("="*80)
print("METHOD 3: POINTWISE 2 PERTURBATION - 80 MEMBERS")
print("="*80)

member_count = 0

for member_idx in range(80):
    seed = BASE_SEED + 2000 + member_idx  # Offset to avoid seed collision
    
    print(f"Member {member_idx:03d}: seed={seed}", end="", flush=True)
    
    try:
        # Initialize generator
        generator = CurrentBasedPerturbationGenerator(
            data_path=DATA_PATH,
            target_idx=TARGET_IDX,
            climato_path=CLIMATO_PATH,
            rand_ds_path=RAND_DS_PATH,
            dt=DT,
            seed=seed,
            is_pointwise=True,
            is_same_noise_in_time=False,
            is_same_noise_for_component=False
        )
        
        # Generate perturbation
        perturbed_state = generator.generate_pointwise_2_perturbation(seed=seed)
        
        # Save to file
        output_path = pointwise2_dir / f"member_{member_idx:03d}_initial_condition.nc"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        ds_out = xr.Dataset({"data": perturbed_state})
        ds_out.to_netcdf(output_path)
        
        print(" ✓ Saved")
        member_count += 1
        
    except Exception as e:
        print(f" ✗ Error: {e}")
        continue
    
    # Progress indicator
    if (member_idx + 1) % 20 == 0:
        print(f"  [{member_count}/80 completed]")

print(f"\n✓ Completed: {member_count} pointwise_2 members generated")

## Summary

In [ ]:
print("="*80)
print("ENSEMBLE GENERATION COMPLETE")
print("="*80)
print(f"Total ensemble members generated: 200")
print(f"  - One vector:    40 members in {onevector_dir}")
print(f"  - Pointwise 1:   80 members in {pointwise1_dir}")
print(f"  - Pointwise 2:   80 members in {pointwise2_dir}")
print()

# Verify generated files
onevector_count = len(list(onevector_dir.glob("member_*/perturbed_initial_condition.nc")))
pointwise1_count = len(list(pointwise1_dir.glob("member_*/perturbed_initial_condition.nc")))
pointwise2_count = len(list(pointwise2_dir.glob("member_*/perturbed_initial_condition.nc")))

print("Verification:")
print(f"  - One vector files:  {onevector_count}/40")
print(f"  - Pointwise 1 files: {pointwise1_count}/80")
print(f"  - Pointwise 2 files: {pointwise2_count}/80")
print(f"  - Total:             {onevector_count + pointwise1_count + pointwise2_count}/200")